# **Оценщики Kernel SHAP: Практика**

Всем привет. В уроке «Как работают ускорения и подсчет SHAP» мы разобрали, как значения Шепли считают не перебором всех $2^n$ коалиций, а по выборке коалиций. Kernel SHAP ([Lundberg и Lee, 2017](https://arxiv.org/abs/1705.07874)) находит значения Шепли как решение взвешенной линейной регрессии: коалиции выбираются с весами ядра Шепли, а коэффициенты регрессии при индикаторах признаков и есть $\phi_i$. В записи [Covert и Lee, 2021](https://arxiv.org/abs/2012.01536) решение этой регрессии с ограничением эффективности имеет вид

$$\phi = A^{-1}\Big(b - \mathbf{1}\,\frac{\mathbf{1}^\top A^{-1} b - \big(v(N) - v(\varnothing)\big)}{\mathbf{1}^\top A^{-1}\mathbf{1}}\Big),\qquad A = \mathbb{E}_z\big[zz^\top\big],\quad b = \mathbb{E}_z\big[z\,\big(v(z) - v(\varnothing)\big)\big],$$

где $z$ — случайная коалиция, записанная вектором из нулей и единиц, а матожидание берется по распределению ядра Шепли. Матрица $A$ зависит только от числа признаков и известна в замкнутом виде, а $b$ приходится оценивать по выборке коалиций. Оригинальный Kernel SHAP оценивает по выборке и $A$, и $b$, поэтому слегка смещен. Unbiased KernelSHAP берет точную $A$ и оценивает только $b$: оценка $\phi$ линейна по $b$ и потому не смещена.

Сегодня посмотрим, что это различие дает на практике, на уже знакомом наборе diabetes и случайном лесе из 300 деревьев; тот же набор и та же модель понадобятся дальше, в уроке «SAGE: Практика». Признаков в diabetes 10, поэтому точный ответ можно посчитать полным перебором 1024 коалиций и мерить ошибку каждого оценщика относительно него.

В этой практике вы:
- посчитаете точные значения Шепли для одной строки теста через `shapiq.ExactComputer` и сверите их с `shap.TreeExplainer`;
- запустите `shap.KernelExplainer`, `shapiq.KernelSHAP` и `shapiq.UnbiasedKernelSHAP` при бюджетах от 64 до 1024 вычислений функции ценности, по 30 сидов на бюджет, и разложите ошибку каждого оценщика на смещение и дисперсию;
- построите доверительные интервалы Unbiased KernelSHAP пакетом `shapley-regression` и проверите, как часто они накрывают точное значение;
- увидите, чего оценщики не исправляют: выбор фона меняет сами точные значения.

**Приятного кодинга!**

Как всегда, сначала установим зависимости. На Colab нет `shapiq` и `shapley-regression`; версии закрепим, чтобы числа совпадали с нашими. `shap` 0.52.0 и `scikit-learn` на Colab уже стоят.

In [ ]:
!pip install shapiq==1.7.0 shapley-regression==0.0.1 -q

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import shapiq
from shapreg import removal, games, shapley, shapley_unbiased

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

# в части сборок numpy матричное умножение печатает ложные предупреждения о переполнении; на числа они не влияют
warnings.filterwarnings('ignore', message='.*encountered in matmul')

## Данные, модель и игра

Набор и модель — те же, что будут в уроке «SAGE: Практика». Набор diabetes встроен в `sklearn`: 442 пациентки и пациента, 10 признаков (возраст, пол, индекс массы тела `bmi`, давление `bp` и шесть показателей крови `s1`–`s6`), цель — количественная мера прогрессии диабета через год. 30% строк отложены на тест, модель — случайный лес из 300 деревьев. Объяснять будем прогноз для первой строки теста.

In [ ]:
X, y = load_diabetes(return_X_y=True, as_frame=True)
feature_names = list(X.columns)
n_players = X.shape[1]

X_train, X_test, y_train, y_test = train_test_split(X.values, y.values, test_size=0.3, random_state=0)
model = RandomForestRegressor(n_estimators=300, max_features=0.33, min_samples_leaf=3,
                              random_state=0, n_jobs=-1).fit(X_train, y_train)

x = X_test[0]
print(f'прогноз для первой строки теста: {model.predict(x[None])[0]:.2f}, истинное значение: {y_test[0]:.0f}')

`shap.KernelExplainer` играет в интервенционную (маргинальную) игру: признаки из коалиции $S$ берутся у объясняемой строки $x$, остальные — из строк фона $B$, и прогнозы усредняются по фону:

$$v(S) = \frac{1}{|B|}\sum_{r\in B} f\big(x_S,\, r_{\bar S}\big).$$

Фон $B$ — 64 случайные строки обучающей выборки. Запишем эту игру функцией, которая принимает сразу пачку коалиций — булеву матрицу размера (число коалиций, 10) — и считает, сколько значений $v(S)$ вычислено. Бюджетом оценщика дальше будет именно это число.

In [ ]:
background = X_train[np.random.default_rng(0).choice(len(X_train), 64, replace=False)]


class Game:
    def __init__(self, model, x, background):
        self.model, self.x, self.background = model, x, background
        self.calls = 0                                         # сколько значений v(S) посчитано

    def __call__(self, coalitions):
        coalitions = np.asarray(coalitions, dtype=bool)
        self.calls += len(coalitions)
        # признаки из S берутся у x, остальные — из каждой строки фона
        rows = np.where(coalitions[:, None, :], self.x[None, None, :], self.background[None, :, :])
        predictions = self.model.predict(rows.reshape(-1, n_players))
        return predictions.reshape(len(coalitions), len(self.background)).mean(axis=1)


game = Game(model, x, background)
v_empty = game(np.zeros((1, n_players)))[0]
v_full = game(np.ones((1, n_players)))[0]
print(f'v(пустая коалиция) = {v_empty:.2f} — средний прогноз по фону')
print(f'v(N) = {v_full:.2f} — прогноз для x, делить между признаками нужно v(N) - v(пустая) = {v_full - v_empty:.2f}')

## Точные значения: `shapiq.ExactComputer`

При 10 признаках коалиций $2^{10} = 1024$, и точные значения Шепли можно посчитать полным перебором. `shapiq.ExactComputer` один раз вычисляет игру на всех коалициях и из этой таблицы считает любой индекс; индекс `'SV'` — значения Шепли.

In [ ]:
game.calls = 0
exact = shapiq.ExactComputer(game, n_players=n_players)('SV')
phi_exact = np.array([exact[(i,)] for i in range(n_players)])

print(f'вычислено значений v(S): {game.calls}')
print(pd.Series(phi_exact, index=feature_names).round(3).sort_values(key=abs, ascending=False).to_string())
print(f'\nсумма phi = {phi_exact.sum():.4f}, v(N) - v(пустая) = {v_full - v_empty:.4f}')

У деревьев есть второй путь к тем же числам. `shap.TreeExplainer` с `feature_perturbation='interventional'` и тем же фоном считает значения Шепли той же игры обходом деревьев, без перебора коалиций. С `feature_perturbation='tree_path_dependent'` и без фона он играет в другую игру: недостающий признак маргинализуется по долям обучающих строк, ушедших в каждую ветвь дерева, а не по фону. Сравним все три столбца.

In [ ]:
tree_interventional = shap.TreeExplainer(model, data=background,
                                         feature_perturbation='interventional').shap_values(x[None])[0]
tree_path = shap.TreeExplainer(model, feature_perturbation='tree_path_dependent').shap_values(x[None])[0]

trees = pd.DataFrame({'ExactComputer': phi_exact, 'Tree interventional': tree_interventional,
                      'Tree path-dependent': tree_path}, index=feature_names)
print(trees.round(3).to_string())
print(f'\nmax |ExactComputer - interventional| = {np.abs(phi_exact - tree_interventional).max():.4f}, '
      f'сумма interventional - (v(N) - v(пустая)) = {tree_interventional.sum() - (v_full - v_empty):.4f}')
print(f'max |ExactComputer - path-dependent| = {np.abs(phi_exact - tree_path).max():.4f}')

**Вопрос для самопроверки.** Какой признак получил наибольшее по модулю точное значение Шепли и чему оно равно? Насколько расходятся `ExactComputer` и interventional `TreeExplainer`, и почему у path-dependent расхождение больше?

<details><summary>Проверить себя</summary>

Наибольшее по модулю точное значение у `bmi`: 48.178, за ним `bp` (25.641) и `s3` (10.939). Сумма десяти значений 95.2309 совпадает с $v(N)-v(\varnothing) = 243.63-148.40$: эффективность выполняется до последнего напечатанного знака. Interventional `TreeExplainer` отличается от `ExactComputer` не больше чем на 0.038 (у `s3`: 10.901 против 10.939), и расходится здесь сам `TreeExplainer`: сумма его значений отличается от $v(N)-v(\varnothing)$ на 0.0104. Path-dependent `TreeExplainer` расходится до 4.159 (у `s5`: −2.017 против 2.143), потому что считает значения Шепли другой игры: недостающий признак он маргинализует по обучающим строкам в ветвях деревьев, а не по 64 строкам фона. Это не ошибка ни одного из двух способов, а ответы на два разных вопроса.

</details>

## Три оценщика при разных бюджетах

Теперь забудем, что точный ответ у нас есть, и дадим оценщикам ограниченный бюджет — от 64 до 1024 вычислений $v(S)$:

- **`shap.KernelExplainer(f, background)`** играет в ту же игру, `nsamples` — сколько коалиций он вычислит. Он экономнее учебного алгоритма: сначала полностью перебирает коалиции самых маленьких и самых больших размеров (1 и $n-1$, затем 2 и $n-2$ и так далее), пока на это хватает бюджета, а остаток добирает случайными коалициями, каждый раз вместе с ее дополнением. `l1_reg=False` отключает отбор признаков перед регрессией: при 10 признаках он и так ничего не отбрасывает.
- **`shapiq.KernelSHAP`** — оригинальный Kernel SHAP: и $A$, и $b$ оцениваются по выборке коалиций.
- **`shapiq.UnbiasedKernelSHAP`** — несмещенный вариант Covert и Lee: $A$ точная, по выборке оценивается только $b$.

У обоих оценщиков `shapiq` есть параметр `pairing_trick`: если он включен, каждая коалиция берется вместе со своим дополнением — это пары комплементарных коалиций из урока «Как работают ускорения и подсчет SHAP». Бюджет `shapiq` проверим счетчиком игры. `KernelExplainer` вызывает модель сам, поэтому обернем модель так, чтобы она считала строки, и поделим их число на размер фона.

In [ ]:
budgets = [64, 128, 256, 512, 1024]
n_seeds = 30


class CountingModel:
    def __init__(self, model):
        self.model, self.rows = model, 0

    def __call__(self, rows):
        self.rows += len(rows)
        return self.model.predict(rows)


def kernel_explainer(budget, seed):
    np.random.seed(seed)                                       # KernelExplainer берет случайные коалиции из np.random
    f = CountingModel(model)
    explainer = shap.KernelExplainer(f, background)
    f.rows = 0                                                 # не считаем вызовы при создании объяснителя
    values = explainer.shap_values(x[None], nsamples=budget, l1_reg=False, silent=True)[0]
    return values, f.rows / len(background)


def shapiq_estimator(approximator_class, pairing):
    def estimate(budget, seed):
        approximator = approximator_class(n=n_players, pairing_trick=pairing, random_state=seed)
        values = approximator.approximate(budget, game)
        return np.array([values[(i,)] for i in range(n_players)])
    return estimate


estimators = {
    'shap.KernelExplainer': lambda budget, seed: kernel_explainer(budget, seed)[0],
    'shapiq.KernelSHAP': shapiq_estimator(shapiq.KernelSHAP, pairing=False),
    'shapiq.KernelSHAP, пары': shapiq_estimator(shapiq.KernelSHAP, pairing=True),
    'shapiq.UnbiasedKernelSHAP': shapiq_estimator(shapiq.UnbiasedKernelSHAP, pairing=False),
    'shapiq.UnbiasedKernelSHAP, пары': shapiq_estimator(shapiq.UnbiasedKernelSHAP, pairing=True),
}

for budget in budgets:
    game.calls = 0
    estimators['shapiq.UnbiasedKernelSHAP'](budget, 0)
    print(f'бюджет {budget:>4}: KernelExplainer вычислил {kernel_explainer(budget, 0)[1]:.0f} значений v(S), '
          f'UnbiasedKernelSHAP — {game.calls}')

Запустим каждый оценщик при каждом бюджете с 30 сидами: 5 оценщиков, 5 бюджетов, 150 запусков на оценщик. На CPU Colab это занимает около минуты.

In [ ]:
runs = {(name, budget): np.array([estimate(budget, seed) for seed in range(n_seeds)])
        for name, estimate in estimators.items() for budget in budgets}
print({key: value.shape for key, value in list(runs.items())[:2]})

## Задание 1. Смещение и дисперсия

Ошибку оценщика при данном бюджете меряем суммарной среднеквадратичной ошибкой по 10 признакам, усредненной по сидам: $\mathrm{MSE} = \frac{1}{30}\sum_{s}\sum_i \big(\hat\phi_i^{(s)} - \phi_i\big)^2$. Она раскладывается на две части:

$$\mathrm{MSE} = \underbrace{\sum_i \big(\bar\phi_i - \phi_i\big)^2}_{\text{смещение}^2} + \underbrace{\sum_i \frac{1}{30}\sum_s \big(\hat\phi_i^{(s)} - \bar\phi_i\big)^2}_{\text{дисперсия}},$$

где $\bar\phi_i$ — среднее оценок признака $i$ по 30 сидам. Соберите таблицу `summary` с индексом (оценщик, бюджет) и столбцами `MSE`, `bias2` и `variance`. Проверьте, что `MSE` равна сумме двух других столбцов.

Учтите одно: даже у несмещенного оценщика `bias2`, посчитанный по 30 сидам, не ноль. Среднее 30 оценок само случайно, и его квадрат отклонения в среднем равен дисперсии, деленной на 30. Поэтому смещение видно, только если `bias2` больше `variance / 30`; добавьте в таблицу и этот столбец.

In [ ]:
rows = []
for (name, budget), estimates in runs.items():
    mse = # Ваш код здесь
    bias2 = # Ваш код здесь
    variance = # Ваш код здесь
    rows.append({'estimator': name, 'budget': budget, 'MSE': mse, 'bias2': bias2, 'variance': variance,
                 'variance / 30': variance / n_seeds})
summary = pd.DataFrame(rows).set_index(['estimator', 'budget'])

print('max |MSE - bias2 - variance| =', np.abs(summary.MSE - summary.bias2 - summary.variance).max())
summary.round(3)

Нарисуем MSE против бюджета в логарифмических осях. Ошибка Монте-Карло убывает как $1/\text{бюджет}$, на таком графике это прямая с наклоном $-1$; серая пунктирная линия показывает этот наклон. При бюджете 1024 оценщики перебирают все коалиции, ошибка нулевая, и на логарифмической оси этой точки нет.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
for name in estimators:
    mse = summary.loc[name].MSE.loc[budgets[:4]]                # при бюджете 1024 ошибка нулевая: на логарифмической оси ее нет
    ax.plot(mse.index, mse.values, marker='o', label=name)
ax.plot(budgets[:4], [1000 * budgets[0] / b for b in budgets[:4]], '--', color='gray', label='slope -1')
ax.set_xscale('log', base=2)
ax.set_yscale('log')
ax.set_xticks(budgets[:4], budgets[:4])
ax.set_xlabel('budget: number of v(S) evaluations')
ax.set_ylabel('MSE against the exact values (30 seeds)')
ax.set_title('Kernel SHAP estimators, random forest on diabetes')
ax.legend()
plt.show()

**Вопрос для самопроверки.** Какой оценщик точнее всех при бюджете 64 и чему равна его MSE? Во сколько раз MSE `shapiq.UnbiasedKernelSHAP` с парами больше, чем у `shapiq.KernelSHAP` с парами, при бюджете 256? Видно ли смещение у оригинального Kernel SHAP, и что происходит при бюджете 1024?

<details><summary>Проверить себя</summary>

При бюджете 64 точнее всех `shapiq.KernelSHAP` с парами: MSE 1.272. У `shap.KernelExplainer` при том же бюджете 5.076, у `shapiq.KernelSHAP` без пар 17.974. При бюджете 256 MSE `shapiq.UnbiasedKernelSHAP` с парами равна 127.451, у `shapiq.KernelSHAP` с парами 0.154 — примерно в 830 раз меньше. При одинаковом выборе пар MSE несмещенного оценщика больше, чем у оригинального, в 80–830 раз при всех бюджетах от 64 до 512. Пары комплементарных коалиций снижают MSE обоим: у `shapiq.KernelSHAP` при бюджете 256 с 2.359 до 0.154, у `shapiq.UnbiasedKernelSHAP` с 242.759 до 127.451.

Смещение оригинального Kernel SHAP видно, но оно мало: у `shapiq.KernelSHAP` без пар при бюджете 64 `bias2` = 0.866 против `variance / 30` = 0.570, а сама дисперсия 17.108. У `shapiq.UnbiasedKernelSHAP` `bias2` не выходит за пределы шума: при бюджете 256 он равен 5.688 при `variance / 30` = 7.902. Отсутствие смещения здесь сопровождается дисперсией в 80–830 раз больше. MSE `shap.KernelExplainer` с 64 до 512 вычислений падает в 30 раз при восьмикратном росте бюджета, быстрее наклона −1: чем больше бюджет, тем больше размеров коалиций он перебирает целиком. При бюджете 1024 MSE всех пяти оценщиков равна 0: `KernelExplainer` вычисляет все 1022 коалиции, кроме пустой и полной, а оценщики `shapiq` при бюджете $2^{10}$ вычисляют все коалиции и решают регрессию точно.

</details>

## Доверительные интервалы: `shapley-regression`

`shapiq` возвращает только точечные оценки. Доверительные интервалы и остановка по сходимости есть в пакете авторов Unbiased KernelSHAP — [`shapley-regression`](https://github.com/iancovert/shapley-regression), он импортируется как `shapreg`. В нем `shapley_unbiased.ShapleyRegression` — Unbiased KernelSHAP, а `shapley.ShapleyRegression` — оригинальный Kernel SHAP, оба с парами комплементарных коалиций.

Игру собираем из частей пакета: `removal.MarginalExtension(background, model.predict)` маргинализует выключенные признаки по тому же фону, `games.PredictionGame` фиксирует объясняемую строку. Оценщик набирает коалиции пачками по `batch_size` пар и после каждой пачки пересчитывает значения и их стандартные ошибки $\hat\sigma_i$. Он останавливается, когда наибольшая стандартная ошибка становится меньше доли `thresh` от размаха значений:

$$\frac{\max_i \hat\sigma_i}{\max_i \hat\phi_i - \min_i \hat\phi_i} < \texttt{thresh}.$$

Возьмем `thresh=0.05`: при размахе около 50 это стандартная ошибка меньше 2.5. Пакет берет случайность из `np.random`, поэтому сид задаем через `np.random.seed`.

In [ ]:
prediction_game = games.PredictionGame(removal.MarginalExtension(background, model.predict), x)


def regression_with_intervals(method, seed):
    np.random.seed(seed)
    values, tracking = method(prediction_game, batch_size=32, detect_convergence=True, thresh=0.05,
                              paired_sampling=True, bar=False, return_all=True)
    return values, tracking


unbiased, unbiased_tracking = regression_with_intervals(shapley_unbiased.ShapleyRegression, seed=0)
original, original_tracking = regression_with_intervals(shapley.ShapleyRegression, seed=0)
print(f'Unbiased KernelSHAP остановился после {unbiased_tracking["iters"][-1]} вычислений v(S), '
      f'Kernel SHAP — после {original_tracking["iters"][-1]}')
pd.DataFrame({'точное': phi_exact,
              'Unbiased': unbiased.values, 'Unbiased, 1.96 sigma': 1.96 * unbiased.std,
              'Kernel SHAP': original.values, 'Kernel SHAP, 1.96 sigma': 1.96 * original.std},
             index=feature_names).round(2)

Посмотрим, как у Unbiased KernelSHAP убывает наибольшая стандартная ошибка по мере набора коалиций. У оригинального Kernel SHAP из `shapley.ShapleyRegression` стандартной ошибки по ходу счета нет: он оценивает ее по разбросу пяти независимых подоценок, каждая из 4 пачек, так что первая оценка появляется только после 20 пачек, 1280 вычислений $v(S)$, и при нашем пороге она же последняя. На графике это одна точка. Пунктир — порог остановки: `thresh`, умноженный на размах точных значений.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
largest_std = [np.max(std) for std in unbiased_tracking['std']]
ax.plot(unbiased_tracking['iters'][:len(largest_std)], largest_std, marker='.', label='shapley_unbiased: Unbiased KernelSHAP')
ax.scatter([original_tracking['iters'][-1]], [original.std.max()], color='tab:orange', zorder=3,
           label='shapley: Kernel SHAP, its only error estimate')
ax.axhline(0.05 * (phi_exact.max() - phi_exact.min()), color='gray', linestyle='--', label='stopping threshold')
ax.set_yscale('log')
ax.set_xlabel('values v(S) computed')
ax.set_ylabel('largest standard error')
ax.set_title('Convergence of shapley-regression, first diabetes test row')
ax.legend()
plt.show()

## Задание 2. Накрывают ли интервалы точное значение

Интервал $\hat\phi_i \pm 1.96\,\hat\sigma_i$ обещает накрывать точное $\phi_i$ примерно в 95% случаев. Проверим обещание у Unbiased KernelSHAP: запустите `regression_with_intervals(shapley_unbiased.ShapleyRegression, seed)` для сидов 0–29 и посчитайте, сколько из $30 \times 10 = 300$ интервалов накрыли точное значение (`covered`). Заодно сохраните, сколько вычислений $v(S)$ понадобилось каждому запуску до остановки (`evaluations`).

In [ ]:
covered = 0
evaluations = []
for seed in range(n_seeds):
    # Ваш код здесь

print(f'накрыли точное значение {covered} из {n_seeds * n_players} интервалов, доля {covered / (n_seeds * n_players):.3f}')
print(f'вычислений v(S) до остановки: в среднем {np.mean(evaluations):.0f}, от {min(evaluations)} до {max(evaluations)}')

**Вопрос для самопроверки.** Сколько из 300 интервалов накрыли точное значение? Сколько вычислений $v(S)$ в среднем понадобилось Unbiased KernelSHAP до остановки, сколько — оригинальному Kernel SHAP при сиде 0, и как это сочетается с таблицей задания 1?

<details><summary>Проверить себя</summary>

Точное значение накрыли 287 из 300 интервалов, доля 0.957 — близко к обещанным 95%. До остановки Unbiased KernelSHAP понадобилось в среднем 2114 вычислений $v(S)$ (от 1728 до 2432), а оригинальному Kernel SHAP при сиде 0 — 1280, и его интервалы при этом в 10–90 раз уже: у `bmi` ±0.16 против ±1.65, у `age` ±0.06 против ±5.49. Это та же картина, что в задании 1: дисперсия несмещенной оценки больше, и до того же порога остановки ей нужно больше коалиций. Unbiased KernelSHAP полезен не точностью точечной оценки, а тем, что интервал вокруг несмещенной оценки накрывает точное значение с обещанной частотой, и это проверено выше.

</details>

## Где оценщики не помогают: выбор фона

Все оценщики выше сходятся к одному числу — точному значению Шепли игры с выбранным фоном. Но фон — часть определения игры $v(S)$, и другой фон дает другую игру с другими точными значениями. Посчитаем точные значения для фонов из 8, 64 и 256 случайных строк обучающей выборки, по 5 разных фонов каждого размера, и сравним разброс точных значений между фонами с разбросом оценки `shap.KernelExplainer` при бюджете 512.

In [ ]:
background_spread = {}
for size in [8, 64, 256]:
    exact_by_background = []
    for seed in range(5):
        other_background = X_train[np.random.default_rng(100 + seed).choice(len(X_train), size, replace=False)]
        values = shapiq.ExactComputer(Game(model, x, other_background), n_players=n_players)('SV')
        exact_by_background.append([values[(i,)] for i in range(n_players)])
    exact_by_background = np.array(exact_by_background)
    background_spread[f'фон {size} строк'] = exact_by_background.std(axis=0)
    print(f'фон {size:>3} строк: bmi от {exact_by_background[:, 2].min():.2f} до {exact_by_background[:, 2].max():.2f}')

background_spread['KernelExplainer, бюджет 512'] = runs[('shap.KernelExplainer', 512)].std(axis=0)
pd.DataFrame(background_spread, index=feature_names).round(2)

**Вопрос для самопроверки.** В каких пределах меняется точное значение `bmi` при пяти разных фонах из 8 строк и из 256 строк? Что больше для большинства признаков: стандартное отклонение точных значений между фонами из 64 строк или стандартное отклонение оценки `KernelExplainer` при бюджете 512?

<details><summary>Проверить себя</summary>

При пяти фонах из 8 строк точное значение `bmi` меняется от 32.46 до 55.49, при фонах из 256 строк — от 46.85 до 47.49. Между фонами из 64 строк стандартное отклонение точных значений у 9 из 10 признаков больше, чем стандартное отклонение оценки `KernelExplainer` при бюджете 512 (0.12–0.15): у `bmi` 1.61 против 0.12, у `s5` 1.17 против 0.12; меньше оно только у `sex` (0.09 против 0.12). Значит, при фоне из 64 строк рост бюджета сверх 512 уменьшает ошибку, которая уже меньше разброса из-за выбора фона. Чтобы сузить этот разброс, нужен фон больше, а не больше коалиций.

</details>

## Заключение

Сегодня вы:
- посчитали точные значения Шепли случайного леса для одной строки diabetes полным перебором 1024 коалиций в `shapiq.ExactComputer` и сверили их с interventional `TreeExplainer` (расхождение до 0.038 при значениях до 48.2), а path-dependent `TreeExplainer` считает другую игру и расходится до 4.16;
- сравнили пять оценщиков при бюджетах от 64 до 1024 вычислений $v(S)$: точнее всех `shapiq.KernelSHAP` с парами комплементарных коалиций, смещение оригинального Kernel SHAP видно только при малом бюджете и примерно в 20 раз меньше его дисперсии (`bias2` 0.866 против 17.108 при бюджете 64 без пар), а MSE `shapiq.UnbiasedKernelSHAP` в 80–830 раз больше, чем у оригинального оценщика с тем же выбором пар;
- проверили интервалы Unbiased KernelSHAP из `shapley-regression`: точное значение накрыли 287 из 300;
- увидели, что выбор фона меняет сами точные значения сильнее, чем ошибка `KernelExplainer` при бюджете 512.

На этом наборе вывод такой: для точечной оценки выгоднее оригинальный Kernel SHAP с парами, а Unbiased KernelSHAP стоит брать, когда нужен интервал с проверенным покрытием, и закладывать для него больший бюджет.

До встречи на следующих шагах!